# Gain attribution with floor check

Leak-free predictive training with a shared encoder and SIGReg. **Rule:** the encoder may simplify the target
latent only in directions where the predictor has stopped improving *and* sits near its achievable error floor.
Spec: `docs/gain_attribution_one_pager.pdf`. Code: `gaf/`. Background and design decisions: `README.md`.

**Stages**
1. Data and premise: what the clips look like, whether slow content alone can satisfy SIGReg, how hard each group is to learn.
2. Baselines (`lewm`, `sg`, `ema`): do they keep slow content and drop the rest?
3. Floors (`detached` + floor probes on the baseline encoders): which latent directions sit on an irreducible noise floor?
4. Fixes (`target`, `context`, `both`): keep fast and hard content, still drop noise?
5. Diagnostics: S over time, leak before and after projection.
6. Optional ablations.

Every run is resumable: finished runs in `OUT` are skipped, so after an interruption just re-run the cell.

In [ ]:
%pip install -q -e .
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU (CPU fallback, slow)")

In [ ]:
%matplotlib inline
from IPython.display import display
import pandas as pd
from gaf.env import EnvCfg, GROUPS
from gaf.train import Cfg, ARMS, premise, run_gt
from gaf import analysis as A

OUT = "/workspace/results"   # on the RunPod network volume, survives pod restarts
SEEDS = (0, 1, 2)
WORKERS = 3   # runs at once on the GPU (separate processes; progress in <run_id>/log.txt). 1 = sequential

# ---- the knobs ---------------------------------------------------------------------------
ENV = EnvCfg(
    res=32, T=3,                 # 32x32 RGB, 2 context frames + 1 target
    n_slow=16, slow_rho=0.98,    # slow patterns: Gaussian across clips, OU within a clip
    n_noise=8,                   # noise patterns: fresh every frame
    v_act=0.3,                   # fast sprite: p' = p + v_act * a
    hard_k=2, h_act=0.3,         # hard sprite: q' = k q + h_act * a. h_act sets the plateau: 0.25 ~1500
                                 # steps, 0.3 ~2500, >= 0.4 never learned (check below before raising it)
    a_slow=1.0, a_noise=1.0, a_fast=4.0, a_hard=4.0, pixel_noise=0.05,  # sprite peak vs pattern RMS
)
COMMON = dict(env=ENV, D=32, steps=20_000, batch=256, lr=1e-3, lam=0.1, wd=0.01,
              basis="excess", floor_steps=1500, floor_warm=True)
# D = 32 > n_slow = 16: room for every group, so a fix need not trade slow content for sprites.
# Stage 4b repeats sg / both in the tight regime D = n_slow.
pd.set_option("display.precision", 3)

## 1 Data and premise
Four kinds of content per clip (rows = clips, columns = frames):
**slow** smooth colour patterns (OU, ρ≈1); **fast** white sprite pushed by the action; **hard** red sprite on an
action-driven chaotic circle map; **noise** patterns of the same kind as slow but fresh every frame.
Actions a_t ∈ [−1, 1]² (random policy) act between frames; the predictor sees them, the encoder does not.

In [ ]:
A.show_clips(ENV, n=4);

**Premise.** The failure needs slow content that can fill SIGReg's budget on its own. Below: SIGReg of hand-made
latents of size D. `slow`: the slow factors, zero-padded to D. `slow+copies`: the remaining dims filled with
Gaussianized nonlinear copies of the slow factors (pairwise products), which is what an encoder can do with spare room.
`slow+copies` close to `gaussian` = slow content alone can (nearly) satisfy SIGReg, so nothing forces the encoder to keep
the sprites even when there is room for them.

In [ ]:
display(premise(ENV, D=COMMON["D"]).to_frame().T)

**Learnability on ground truth** (no encoder): a predictor from the true context factors and actions to the true
target factors. Slow should be learned almost instantly, fast quickly, hard more slowly, noise never (R² ≈ 0).
The second table hides the actions: fast and hard must collapse to their cap (≈ 0.25), otherwise they are not really
action-conditioned. `hard` should sit near R² ≈ 0 for a long plateau and then be learned, clearly after `fast`.

**Hard must be learnable, and the probe must be ahead.** If hard is never learned, even from ground truth, no predictor or
probe can learn it and the gate cannot tell it from noise: lower `h_act`. If it is learned, the floor probe must know
that before the predictor does, otherwise it overestimates L* on hard directions and opens the gate while the predictor
is still on its plateau (the probe-bias caveat). The probe is warm-started: it adds `floor_steps` of training per refresh,
and refreshes come every `floor_every` steps, so it is ahead of the predictor whenever `floor_steps > floor_every`.
The cell below computes the step at which the probe has seen as many steps as hard's learn time.

In [ ]:
gt = run_gt(ENV, steps=5000, desc="ground truth, with actions")
gt_noact = run_gt(ENV, steps=5000, actions=False, desc="ground truth, without actions")
ax = gt.plot(figsize=(6, 3.4), color=[A.GROUP_COLOR[g] for g in GROUPS])
ax.set(xlabel="step", ylabel="held-out R²", ylim=(-0.05, 1.02)); ax.legend(loc="lower right")
display(pd.concat({"with actions": gt.iloc[[round(i * (len(gt) - 1) / 4) for i in range(5)]],
                   "without actions": gt_noact.iloc[[-1]]}))
t90 = {g: (int(gt.index[(gt[g] >= 0.9 * gt[g].iloc[-1]).values.argmax()]) if gt[g].iloc[-1] > 0.5
           else None) for g in GROUPS[:3]}   # None: not learned within run_gt's budget
print("steps to 90% of final R² on ground truth:", t90)
c0 = Cfg(**COMMON)
if t90["hard"] is None:
    print("WARNING: hard is not learned within 5000 steps even from ground-truth factors. No predictor or probe can learn "
          "it, so the gate cannot tell it from noise. Lower ENV.h_act (0.25 is learned in ~1500 steps).")
else:
    k = -(-t90["hard"] // c0.floor_steps) - 1                 # refreshes after the first one
    ready = c0.floor_start + k * c0.floor_every               # step at which the warm probe has seen t90 steps
    print(f"warm probe has trained {t90['hard']} steps after refresh {k + 1} (step {ready}); the predictor needs "
          f"~{t90['hard']} steps (more with a moving encoder)")
    print("probe is ahead of the predictor: the floor shows hard is learnable while the predictor is on its plateau"
          if ready < t90["hard"] else
          "WARNING: the probe is not ahead of the predictor. Raise floor_steps or lower floor_every, "
          "otherwise hard directions get too high a floor and the gate opens on them.")

## 2 Baselines
`lewm`: full gradient through both paths. `sg`: stop-grad target (the leak is unchecked). `ema`: EMA target encoder.
Expected: slow content kept, fast and hard content dropped (or never learned), noise dropped.
Every long cell shows a progress bar. The bar of a training stage counts the training steps of all runs still to do, so
its ETA covers the whole cell, and it lists what each running run is doing (`train`, `floor probe`, `eval`, `final probes`).
Each finished run prints its time. Budget: about 4–7 min per run alone; with `WORKERS = 3` runs share the GPU, so each
takes longer but the stage finishes roughly 2–3× sooner. With `WORKERS > 1` the eval lines (R² per group every 500 steps)
go to the log of each run; watch one with `!tail -f /workspace/results/<run_id>/log.txt`.

In [ ]:
A.execute(A.grid(A.BASELINES, SEEDS, **COMMON), OUT, workers=WORKERS)
df = A.collect(OUT)
t_base = A.table(df[df.arm.isin(A.BASELINES) & (df.tag == "")])
display(t_base)
display(A.baseline_verdict(t_base))
A.bars(t_base); A.curves(df, OUT, A.BASELINES);

## 3 Floors
**(a)** `detached`: the encoder sees only SIGReg; the predictor trains on fully detached latents. Its L* history is the
noise floor of a latent that prediction never shaped.
**(b)** Floor probes on the final encoders of the baselines: one fresh predictor per frozen encoder, per-direction
held-out error L*_j, plotted against how much of direction j is noise. Noise directions should sit high, slow ones near 0.

In [ ]:
A.execute(A.grid(["detached"], SEEDS, **COMMON), OUT, workers=WORKERS)
df = A.collect(OUT)
display(A.table(df[df.arm == "detached"]))
cfgs = [c for c in A.grid(["detached", "sg", "lewm"], SEEDS[:1], **COMMON)]
fl = pd.concat(A.floors_of(c, OUT) for c in cfgs)
display(fl.groupby("arm")[["floor", *GROUPS]].mean())
A.floor_scatter(fl);

## 4 Fixes
`target`: open target path on saturated directions only. `context`: leak projection on the context gradient only.
`both`: the full method. Success (vs. `sg`): fast and hard R² up by ≥ 0.3, noise and slow within 0.1 of `sg`.

In [ ]:
A.execute(A.grid(A.FIXES, SEEDS, **COMMON), OUT, workers=WORKERS)
df = A.collect(OUT)
t_all = A.table(df[df.tag == ""])
display(t_all)
display(A.fix_verdict(t_all, ref="sg"))
A.bars(t_all); A.curves(df, OUT, ["sg", *A.FIXES], tag="");

### 4b Tight latent (D = n_slow)
Slow content alone now fills SIGReg's budget exactly, so keeping the sprites must cost slow R²: the verdict allows
slow to drop by up to 0.3. If the fix works here too, it holds without spare capacity. Tagged `tight`.

In [ ]:
D_TIGHT = ENV.n_slow
A.execute(A.grid(["sg", "both"], SEEDS, **{**COMMON, "D": D_TIGHT, "tag": "tight"}), OUT, workers=WORKERS)
df = A.collect(OUT)
display(premise(ENV, D=D_TIGHT).to_frame().T)
t_tight = A.table(A.select(df, tag="tight"))
display(t_tight)
display(A.fix_verdict(t_tight, ref="sg", slow_slack=0.3))
A.bars(t_tight);

## 5 Diagnostics
- `n_S` protected directions; `leak_pos` share of diagnostic steps with ℓ > 0; `leak_cos` / `leak_cos_post` cosine of
  g_tgt,S and the applied context-side AdamW update before / after projection (≈ 0 after); `proj_ratio` norm ratio.
- `lreal_S`, `lreal_open`: exact check ℓreal = L(θ′, z_t(θ)) − L(θ′, z_t(θ′)) summed over S / over the open directions.
  After projection, `lreal_S` should be ≈ 0 (SIGReg and Adam momentum still move the target a little).
- `wb_ratio` within/between-clip variance of z (slow collapse → small), `rank_dz` effective rank of z_t − z_{t−1}.

In [ ]:
display(A.diag_table(df[df.tag == ""]))
A.leak_panel(df, OUT, ["sg", *A.FIXES], tag="");

Per-direction view of one run: which directions are protected, their relative gap and excess over the floor, and
which group each direction carries. In a working run, fast/hard directions stay in S while they are being learned and
noise directions leave S.

In [ ]:
rid = A.select(df, arm="both", seed=0, tag="").run_id.iloc[0]
A.gate_map(OUT, rid);
rid = A.select(df, arm="sg", seed=0, tag="").run_id.iloc[0]
A.gate_map(OUT, rid);

## 6 Ablations (optional)
Each changes one knob of `both`; `tag` keeps them apart in the tables. Uncomment what you need.

In [ ]:
ABL = {
    "proj=grad": dict(proj_on="grad"),      # one-pager sketch: project the raw gradient only
    "open-unprojected": dict(proj_open=False),  # open target path not projected against S
    "basis=dim": dict(basis="dim"),         # gate per latent dim (arbitrary under SIGReg)
    "basis=resid": dict(basis="resid"),     # gate in the floor-residual eigenbasis
    # "alpha=0.5": dict(alpha=0.5),         # partial projection (over-projection caveat)
    # "cold-floor": dict(floor_warm=False), # fresh probe at every refresh
    # "wd=0": dict(wd=0.0),
    # "soft": dict(soft=True),              # soft open-path weights
    # "sgd": dict(opt="sgd", lr=0.05),      # plain SGD: the derivation holds exactly
}
cfgs = [c for name, kw in ABL.items() for c in A.grid(["both"], SEEDS, **{**COMMON, **kw, "tag": name})]
A.execute(cfgs, OUT, workers=WORKERS)
df = A.collect(OUT)
abl = df[(df.arm == "both") & df.tag.isin(["", *ABL])]
display(abl.groupby("tag")[A.R2 + ["n_S", "leak_cos_post", "lreal_S"]].mean())

## Environment sweeps (optional)
The failure and the fix depend on how salient and how hard the fast content is. Example: harder chaotic map and a
fainter fast sprite. Results go to a separate folder so tables stay comparable.

In [ ]:
# ENV2 = EnvCfg(**{**ENV.__dict__, "hard_k": 3, "a_fast": 2.0})   # harder map, fainter sprite
# ENV2 = EnvCfg(**{**ENV.__dict__, "n_slow": 24})                   # less spare room at D=32
# OUT2 = OUT + "/hard3"
# A.execute(A.grid(["sg", "both"], SEEDS, **{**COMMON, "env": ENV2}), OUT2, workers=WORKERS)
# display(A.table(A.collect(OUT2)))

## 7 Why are fast and hard lost? (one seed)
In the baselines the sprites are gone by step 500 in every arm that shares the encoder between context and target
(`sg`, `context`, `target`, `both`). `detached` keeps them (fast R² 0.33 at step 500), and `ema` keeps and learns fast in
two of three seeds. Hypothesis: a race. The predictor needs about 700 steps (fast) and 2100 steps (hard) to learn a sprite.
Until then, sprite variance in the context latents is noise for it. The context-path gradient, which the projection leaves
alone as legitimate learning, prunes the sprites from the shared encoder; the target loses them too, and nothing is left
to learn.

All runs use seed 0 and also write early evals (R² per group at steps 0, 25, 50, …). 7 runs, about 12 min with 3 workers.

| Group | Runs | Question | If the hypothesis holds |
|---|---|---|---|
| **early** (3000 steps) | `sg`, `ema`, `detached`, `both` | When and how fast do the sprites vanish? | fast and hard fall below 0.1 within the first ~100–300 steps in `sg` and `both`, long before the floor exists (step 500); `ema` and `detached` keep them |
| **head start** (8000 steps) | `sg`, `both`; the encoder gets only SIGReg for the first 3000 steps while the predictor trains | Does a head start for the predictor keep the sprites? | fast (maybe hard) is learned during the head start and survives the switch at step 3000 |
| **block** (20000 steps) | `block`: the encoder gets the prediction loss only in the open directions, through both the context and the target path; the predictor still trains on all directions | Does blocking the context part too, wherever the target part is blocked, keep the sprites and still drop noise? | fast (maybe hard) survives like in `ema`, noise is dropped, slow is kept |

The exact leak is split by source in `lk_ctx_*` (context part of the step) and `lk_rest_*` (SIGReg and open target path),
summed over the protected (`_S`) and the open directions, predictor held fixed. Positive = that part makes the target
easier to predict.

In [ ]:
EARLY = (0, 25, 50, 100, 150, 200, 300, 400, 500, 750, 1000, 1500, 2000, 2500)
EARLY_HS = EARLY + (3000, 3025, 3050, 3100, 3200, 3400, 4000)   # dense around the switch at step 3000


def cfg0(arm, **kw):
    return Cfg(arm=arm, seed=0, **{**COMMON, **kw})


WHY = (
    [cfg0(a, steps=3000, early_eval=EARLY, tag="early") for a in ("sg", "ema", "detached", "both")]
    + [cfg0(a, steps=8000, early_eval=EARLY_HS, head_start=3000, tag="headstart") for a in ("sg", "both")]
    + [cfg0("block", steps=20_000, early_eval=EARLY, tag="block")]
)
A.execute(WHY, OUT, workers=WORKERS)
df = A.collect(OUT)

**7a When do the sprites vanish?** Ridge R² from z_t at the early steps (step 0 is the random initialisation).
The floor probe only starts at step 500.

In [ ]:
R_EARLY = {k: dict(arm=k, tag="early") for k in ("sg", "ema", "detached", "both")}
display(A.snapshot(df, OUT, R_EARLY))
A.trace(df, OUT, R_EARLY);

**7b Head start.** The encoder gets only SIGReg until step 3000; the predictor trains on the (detached) latents.
Then the arm switches on. Plain `sg` and `both` are the 20k-step runs of stage 2 and 4 (seed 0).

In [ ]:
R_HS = {"sg": dict(arm="sg", tag="", seed=0), "sg + head start": dict(arm="sg", tag="headstart"),
        "both": dict(arm="both", tag="", seed=0), "both + head start": dict(arm="both", tag="headstart")}
display(A.snapshot(df, OUT, R_HS, steps=(500, 2000, 3000, 3050, 3200, 4000, 5000, 8000),
                   cols=("r2_fast", "r2_hard", "r2_noise")))
A.trace(df, OUT, R_HS, cols=("r2_fast", "r2_hard", "r2_noise"), xlog=False);

In [ ]:
R_BLOCK = {"sg": dict(arm="sg", tag="", seed=0), "both": dict(arm="both", tag="", seed=0),
           "ema (seed 0)": dict(arm="ema", tag="", seed=0), "block": dict(arm="block", tag="block")}
display(A.table(df[df.tag == "block"]))
display(A.snapshot(df, OUT, R_BLOCK, steps=(500, 1000, 3000, 10000, 20000),
                   cols=("r2_fast", "r2_hard", "r2_noise", "r2_slow", "n_S")))
A.trace(df, OUT, R_BLOCK, xlog=False);

**7c Block.** `block` against the baselines of stage 2 and 4 (seed 0): the encoder is shaped by the prediction loss
only where the gate says the predictor has finished learning. `n_S` is the number of protected directions.

**7d Where does the exact leak come from?** If `lk_ctx_S` is about 0 after projection while the loss is still lowered
through the target, the remaining leak is `lk_rest_S` (SIGReg), which the method never projects.

In [ ]:
display(A.snapshot(df, OUT, {"sg": dict(arm="sg", tag="early"), "both": dict(arm="both", tag="early"),
                    "block": dict(arm="block", tag="block")},
                   steps=(500, 1000, 1500, 2000, 3000),
                   cols=("lk_ctx_S", "lk_rest_S", "lk_ctx_open", "lk_rest_open")))

**Reading the results**
- Sprites already low at step 25–100 in `sg` and `both`: pruning happens in the first Adam steps, before any gate or floor
  can exist. Only a head start or a lagged target can help.
- They vanish between 100 and 500: the race window. A floor probe from step 100 and protection from the start might help.
- Fast survives the head start and holds after the switch: race confirmed. The remedy is a schedule, not a better gradient rule.
- Fast collapses within about 100 steps of the switch although the predictor already knows it: pruning is not about what
  the predictor can use; the shared target itself is the problem (try anchoring S to a lagged encoder).
- `lk_rest_S` > 0 with `lk_ctx_S` ≈ 0: the leak comes from SIGReg, not from the context step.
- `lk_ctx_S` > 0 after projection: the first-order projection is not enough.
- `block` keeps fast but loses noise handling (noise stays high): the gate keeps directions protected that are only noise;
  check `n_S` and the floors. `block` loses the sprites anyway: masking the loss gradient is not enough, because the
  encoder step for the open directions still moves the protected ones through the shared weights (see `lk_ctx_S`).

**7e Hard sprite, seed 1.** In seed 0 the SIGReg-only encoder never builds hard content (`detached`: ridge R² 0.005
at step 2000), so there was nothing to protect. In seed 1 it does (0.27 at step 2000, 0.32 at step 5000). Here the head
start is 5000 steps (the predictor needs about 2100–3000 steps to learn hard even from clean factors), and `block` runs
the full 20k steps. 3 runs, about 12 min with 3 workers. The comparison runs are the v1 runs of seed 1 (`detached`, `sg`,
`both`, `ema`).

What to look for in `r2_hard`:
- It should be about 0.3 during the head start (availability), then rise towards 0.9+ after the switch at step 5000.
- Stays near 0.3: the predictor has not learned hard yet; lengthen the head start or the run.
- Decays after the switch: hard is pruned before the predictor can use it, as fast was in seed 0.
- Never above 0.05: hard is not available in this seed either; check `detached` in the first plot.

In [ ]:
EARLY_S1 = (0, 50, 100, 200, 300, 500, 750, 1000, 1500, 2000, 3000, 4000, 5000, 5050, 5100, 5200, 5400,
            6000, 7000, 8000)


def cfg_s(arm, seed, **kw):
    return Cfg(arm=arm, seed=seed, **{**COMMON, **kw})


S1 = (
    [cfg_s(a, 1, steps=12_000, early_eval=EARLY_S1, head_start=5000, tag="s1-hs5000") for a in ("sg", "both")]
    + [cfg_s("block", 1, steps=20_000, early_eval=EARLY_S1, tag="s1-block")]
)
A.execute(S1, OUT, workers=WORKERS)
df = A.collect(OUT)

In [ ]:
R_H1 = {"detached (v1)": dict(arm="detached", tag="", seed=1), "sg (v1)": dict(arm="sg", tag="", seed=1),
        "sg + head start": dict(arm="sg", tag="s1-hs5000"), "both + head start": dict(arm="both", tag="s1-hs5000")}
display(A.snapshot(df, OUT, R_H1, steps=(500, 2000, 3000, 5000, 5100, 5400, 6000, 8000, 12000),
                   cols=("r2_hard", "r2_fast")))
A.trace(df, OUT, R_H1, cols=("r2_hard", "r2_fast", "r2_noise", "r2_slow"), xlog=False);

In [ ]:
R_B1 = {"sg (v1)": dict(arm="sg", tag="", seed=1), "both (v1)": dict(arm="both", tag="", seed=1),
        "ema (v1)": dict(arm="ema", tag="", seed=1), "block": dict(arm="block", tag="s1-block")}
display(A.snapshot(df, OUT, R_B1, steps=(500, 2000, 5000, 10000, 20000),
                   cols=("r2_hard", "r2_fast", "r2_noise", "r2_slow", "n_S")))
A.trace(df, OUT, R_B1, cols=("r2_hard", "r2_fast", "r2_noise", "r2_slow"), xlog=False)
final = df[df.tag.isin(["s1-hs5000", "s1-block"])]
display(final[["arm", "tag"] + [f"r2_{g}" for g in GROUPS] + [f"mlp_r2_{g}" for g in GROUPS]].round(3))

**7f Why does hard appear so late? Two checks.** Hypothesis: (1) hard enters the SIGReg-shaped latent late and
unreliably, perhaps because the red (bipolar) sprite looks less salient to the encoder than the white one; (2) even when it is
there, a predictor needs far more steps to learn it than the encoder leaves it alone.

- **Colour swap:** `detached` runs (encoder trained by SIGReg only, 3 seeds, 3000 steps) with the colours swapped: the
  fast sprite is red, the hard sprite white. Both still carry the same energy. In these runs `r2_hard` is the *white* sprite
  with the hard dynamics. If it now appears early and `r2_fast` (red) late, availability is a colour effect.
- **Learn time in latent space:** `detached`, seed 1, 5000 steps (hard content present, ridge R² about 0.3). Its encoder is
  then frozen and a fresh predictor is trained on its latents for 20k steps. The curve of `hard` is the number of predictor
  steps needed, which a head start has to exceed. `fast` is the reference. 4 runs, about 5 min, then about 3 min of training
  for the predictor on the GPU.

In [ ]:
def cfg_s(arm, seed, **kw):
    return Cfg(arm=arm, seed=seed, **{**COMMON, **kw})


ENV_SWAP = EnvCfg(**{**ENV.__dict__, "swap_colors": True})   # fast sprite red, hard sprite white
FREEZE = cfg_s("detached", 1, steps=5000, tag="freeze5000")   # seed-1 encoder after 5000 SIGReg-only steps
A.execute([cfg_s("detached", s, env=ENV_SWAP, steps=3000, tag="swap") for s in (0, 1, 2)] + [FREEZE],
          OUT, workers=WORKERS)
df = A.collect(OUT)

In [ ]:
rows = {f"{lab} s{s}": dict(arm="detached", tag=tag, seed=s)
        for lab, tag in (("v1", ""), ("swap", "swap")) for s in (0, 1, 2)}
display(A.snapshot(df, OUT, rows, steps=(500, 1000, 2000, 3000), cols=("r2_hard", "r2_fast")))
A.trace(df, OUT, {"v1 (mean of 3 seeds)": dict(arm="detached", tag=""),
                  "colours swapped (mean of 3)": dict(arm="detached", tag="swap")},
        cols=("r2_hard", "r2_fast"), xlog=False);

In [ ]:
lat = A.latent_learning(FREEZE, OUT, steps=20_000, eval_every=500,
                        desc="predictor on frozen latents (seed 1, 5000 SIGReg-only steps)")
print("availability in the latent (ridge R² from z_t):", {g: round(v, 3) for g, v in lat.attrs["availability"].items()})
ax = lat[list(GROUPS)].plot(figsize=(6, 3.4), color=[A.GROUP_COLOR[g] for g in GROUPS])
ax.set(xlabel="predictor steps", ylabel="held-out R² of the prediction", ylim=(-0.05, 1.02))
ax.legend(loc="lower right")
t90 = {g: (int(lat.index[(lat[g] >= 0.9 * lat[g].iloc[-1]).values.argmax()]) if lat[g].iloc[-1] > 0.3 else None)
       for g in ("fast", "hard")}   # None: not learned within the budget
print("predictor steps to 90% of the final R² (None: not learned):", t90)
display(lat.iloc[[round(i * (len(lat) - 1) / 6) for i in range(7)]].round(3))

**Reading 7f**
- Colour swap: the white `hard` sprite appears early (R² about 0.4–0.5 by step 500) and the red `fast` sprite late: hard's
  lateness is a salience effect of its colour, not of its dynamics. Fix by making the sprites equally salient.
- Both sprites stay as before (fast early, hard late): not colour; look at the dynamics or at the seed.
- Seed 2 has no fast content in v1: if the swap gives it a sprite there, the seed effect is also about colour and position.
- Learn time: `hard` plateaus and rises after N predictor steps. A head start has to exceed N, on top of the SIGReg-only
  steps the encoder needs to build hard in the first place (about 2000–5000 in seed 1).
- `hard` stays near 0 although its availability is above 0.2: the information is in the latent but not accessible to this
  predictor (nonlinear, needs more steps or capacity), and a head start alone cannot fix it.

**7g Is hard lost because of colour, or because of the doubling map?** Findings so far (seed 1, head start 5000): hard is
available (0.32), the predictor learns it, yet it is pruned within about 50–400 steps after the switch, while fast, at a
similar predictive R², is reinforced to 0.99. Two candidate reasons, tested here with a head start of 5000 steps (12k steps):

| Variant | Runs | What it separates |
|---|---|---|
| **white hard sprite**, doubling map (`k=2`) | `both` (seeds 0, 1) and `sg` (seed 1) | colour: the sprite is as salient as fast was, so availability is no longer the handicap |
| **white hard sprite, `k=1`** (a plain shift like fast) | `both` and `sg`, seed 1 | steepness: the doubling map amplifies errors in the encoded position by 2; with `k=1` it does not |

Comparison runs: the 7e runs (red hard sprite, `k=2`). 5 runs, about 15 min with 3 workers. `r2_hard` is the hard sprite
in every case (white here), `pred_r2_hard` its predictability from the predictor output.

In [ ]:
EARLY_W = (0, 50, 100, 200, 300, 500, 750, 1000, 1500, 2000, 3000, 4000, 5000, 5050, 5100, 5200, 5400,
           6000, 7000, 8000)


def cfg_s(arm, seed, **kw):
    return Cfg(arm=arm, seed=seed, **{**COMMON, **kw})


ENV_W = EnvCfg(**{**ENV.__dict__, "swap_colors": True})                # white hard sprite, k = 2
ENV_W1 = EnvCfg(**{**ENV.__dict__, "swap_colors": True, "hard_k": 1})  # white hard sprite, k = 1
hs = dict(steps=12_000, early_eval=EARLY_W, head_start=5000)
G7 = (
    [cfg_s("both", s, env=ENV_W, tag="w-hs5000", **hs) for s in (0, 1)]
    + [cfg_s("sg", 1, env=ENV_W, tag="w-hs5000", **hs)]
    + [cfg_s(a, 1, env=ENV_W1, tag="w-k1-hs5000", **hs) for a in ("both", "sg")]
)
A.execute(G7, OUT, workers=WORKERS)
df = A.collect(OUT)

In [ ]:
R_G = {"both+hs, red hard (7e)": dict(arm="both", tag="s1-hs5000"),
       "both+hs, white hard": dict(arm="both", tag="w-hs5000", seed=1),
       "both+hs, white hard, k=1": dict(arm="both", tag="w-k1-hs5000"),
       "sg+hs, white hard": dict(arm="sg", tag="w-hs5000")}
steps = (500, 2000, 3000, 5000, 5050, 5100, 5400, 6000, 8000, 12000)
for col in ("r2_hard", "pred_r2_hard", "n_S"):
    print(col)
    display(A.snapshot(df, OUT, R_G, steps=steps, cols=(col,)).rename(columns=lambda c: c.split("@")[1]))
A.trace(df, OUT, R_G, cols=("r2_hard", "pred_r2_hard", "r2_fast", "n_S"), xlog=False);

In [ ]:
display(A.snapshot(df, OUT, {"both+hs, white hard, seed 0": dict(arm="both", tag="w-hs5000", seed=0),
                             "both+hs, white hard, seed 1": dict(arm="both", tag="w-hs5000", seed=1)},
                   steps=(500, 2000, 5000, 5100, 5400, 8000, 12000), cols=("r2_hard", "pred_r2_hard")))
final = df[df.tag.isin(["w-hs5000", "w-k1-hs5000"])]
display(final[["arm", "tag", "seed"] + [f"r2_{g}" for g in GROUPS] + [f"mlp_r2_{g}" for g in GROUPS]].round(3))

**Reading 7g**
- White hard sprite with `k=2` survives the switch and `r2_hard` climbs towards 0.9: colour (availability) was the main
  problem. With a sprite as salient as fast, head start plus `both` already handles hard.
- White hard sprite with `k=2` is pruned like the red one, but with `k=1` it survives: the doubling map is the problem. The
  prediction gradient prunes what the latent encodes too imprecisely for the predictor to track.
- Pruned in both variants: neither colour nor steepness; look at the interaction with the action or at the gate (`n_S`).
- `n_S` is 0 before the switch in every run: the gate declares the sprite saturated while `pred_r2_hard` is still rising.
  `sg` (no gate) against `both` shows how much of the pruning is the open target path.
- Seeds 0 and 1 disagree: availability differs by seed (seed 2 has no sprites at all); do not over-read a single seed.

## 8 Reference-gated retention (unsupervised)
A general fix that needs no labels, no groups and no hand-set head start. Full description: `docs/reference_gated_retention.md`.

A **shadow encoder** (trained by SIGReg and a reconstruction decoder only, never by the prediction loss) keeps all content.
A **shadow predictor** on its latents gives a reference floor: per direction, how much is predictable in principle (ρ). The
live encoder gets a **retention loss**: a linear head must read the shadow latent's retained part from the live latent. A
direction is retained while the shadow predictor is still improving (protect by default), or while it is predictable and the
live predictor is still behind the floor; it is free when it is settled and unpredictable (noise). `retain_open` adds the
floor-gated open target path of `both` to remove noise; `retain` is retention alone.

| Runs (12k steps, no head start) | Environment | Question |
|---|---|---|
| `retain_open`, seeds 0 and 1 | default (red hard sprite) | does it keep fast without a head start, as `both` + head start did? |
| `retain_open`, seeds 0 and 1 | white hard sprite | does it keep hard, including seed 1 where every earlier fix failed? |
| `retain_open`, seed 1, `shadow_rec=0` | white hard sprite | is the reconstruction needed for availability (SIGReg-only shadow)? |
| `retain`, seed 1 | white hard sprite | retention alone (noise should stay) |

6 runs, about 30 min with 3 workers (each run trains a second encoder and a decoder). Success criteria, fixed before running:
- fast ≥ 0.9 (default env) and hard ≥ 0.7 (white env) in both seeds, noise ≤ 0.3, slow ≥ 0.9, without any head start;
- the unsupervised ρ separates content from noise: `ref_rho_pred − ref_rho_noise ≥ 0.5` by step 2000.

In [ ]:
def cfg_s(arm, seed, **kw):
    return Cfg(arm=arm, seed=seed, **{**COMMON, **kw})


ENV_W = EnvCfg(**{**ENV.__dict__, "swap_colors": True})   # white hard sprite
R8 = (
    [cfg_s("retain_open", s, env=env, steps=12_000, tag=tag)
     for s, env, tag in [(0, ENV, "ret-red"), (1, ENV, "ret-red"), (0, ENV_W, "ret-white"), (1, ENV_W, "ret-white")]]
    + [cfg_s("retain_open", 1, env=ENV_W, steps=12_000, shadow_rec=0.0, tag="ret-white-norec")]
    + [cfg_s("retain", 1, env=ENV_W, steps=12_000, tag="ret-only-white")]
)
A.execute(R8, OUT, workers=WORKERS)
df = A.collect(OUT)

In [ ]:
cols = ["arm", "tag", "seed"] + [f"r2_{g}" for g in GROUPS] + [f"mlp_r2_{g}" for g in GROUPS]
ours = df[df.tag.str.startswith("ret-")]
best = df[(df.arm == "both") & df.tag.isin(["s1-hs5000", "w-hs5000"])]   # best earlier fix: both + head start
base = df[(df.arm == "sg") & (df.tag == "") & df.seed.isin([0, 1])]
print("retention (no head start), against both + head start and sg")
display(pd.concat([ours, best, base])[cols].sort_values(["tag", "seed"]).round(3))

In [ ]:
ref_cols = ["ref_rho_noise", "ref_rho_pred", "ref_w", "ref_unsettled", "ref_free"] + [f"sh_r2_{g}" for g in GROUPS]
print("what the reference knows (sh_r2_*: the shadow latent itself) and whether rho separates content from noise")
display(ours[["tag", "seed"] + ref_cols].sort_values(["tag", "seed"]).round(3))
R_REF = {f"{t} s{s}": dict(arm="retain_open", tag=t, seed=s) for t in ("ret-red", "ret-white") for s in (0, 1)}
display(A.snapshot(df, OUT, R_REF, steps=(500, 1000, 2000, 5000, 12000), cols=("ref_rho_pred", "ref_rho_noise")))

In [ ]:
R_8 = {"sg (v1), seed 1": dict(arm="sg", tag="", seed=1),
       "both + head start, white, seed 1": dict(arm="both", tag="w-hs5000", seed=1),
       "retain_open, white, seed 1": dict(arm="retain_open", tag="ret-white", seed=1),
       "retain_open, SIGReg-only shadow": dict(arm="retain_open", tag="ret-white-norec")}
A.trace(df, OUT, R_8, cols=("r2_hard", "r2_fast", "r2_noise", "r2_slow"), xlog=False);
R_8b = {"sg (v1), seed 0": dict(arm="sg", tag="", seed=0),
        "both + head start, white, seed 0": dict(arm="both", tag="w-hs5000", seed=0),
        "retain_open, white, seed 0": dict(arm="retain_open", tag="ret-white", seed=0)}
A.trace(df, OUT, R_8b, cols=("r2_hard", "r2_fast", "r2_noise", "r2_slow"), xlog=False);

**Reading stage 8**
- Criteria met in both seeds and both environments: the method does what a hand-set head start did, unsupervised, and for hard
  in seed 1 where the head start failed. Next: more seeds, then a real benchmark.
- `ref_rho_pred − ref_rho_noise` large but retention fails: the weights are right and the retention is too weak or too
  short-lived (raise `ret_lam`, lower `ret_eps_f`), or the live encoder cannot carry both (capacity).
- `sh_r2_hard` / `sh_r2_fast` near 0: the shadow does not have the sprite either, so nothing can be retained. Availability is
  still the limit; compare the SIGReg-only shadow (`shadow_rec=0`) against the reconstruction shadow.
- Reconstruction shadow clearly better than SIGReg-only: the reconstruction is what guarantees availability.
- `retain` keeps noise (≈ 0.9) and `retain_open` drops it: noise removal still comes from the open path.
- ρ does not separate (small gap, or unstable over steps): the shadow predictor is not settled in time; lengthen `ret_lag`
  or `ret_tau`, or the shadow needs more capacity or steps.
- Fast and hard retained but slow suffers: the retention costs live capacity; compare slow with `both` + head start.

## All runs

In [ ]:
df = A.collect(OUT)
df.to_csv(f"{OUT}/summary.csv", index=False)
print(len(df), "runs;", f"{df.seconds.sum() / 3600:.1f} GPU-hours")